In [1]:
###### Hello World test ######
from PyQt6.QtWidgets import QApplication, QWidget

# Only needed for access to command line arguments
import sys

# You need one (and only one) QApplication instance per application.
# Pass in sys.argv to allow command line arguments for your app.
# If you know you won't use command line arguments QApplication([]) works too.
app = QApplication(sys.argv)

# Create a Qt widget, which will be our window.
window = QWidget()
window.show()  # IMPORTANT!!!!! Windows are hidden by default.

# Start the event loop.
app.exec()


# Your application won't reach here until you exit and the event
# loop has stopped.



0

In [2]:
###### Basic Window ######
import sys
from PyQt6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget, QPushButton, QSlider, QLabel, QLineEdit, QHBoxLayout
from PyQt6.QtCore import Qt
import vtkmodules.all as vtk
from vtk.qt.QVTKRenderWindowInteractor import QVTKRenderWindowInteractor

class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super(MainWindow, self).__init__(parent)
        
        self.setWindowTitle("VTK with PyQt6")

        self.frame = QWidget()
        self.layout = QVBoxLayout()

        # Set the window size to be 800x600 pixels and position it at 100, 100
        self.setGeometry(100, 100, 800, 600)
        
        # Initializing the VTK Render Widget into the Qt layout
        self.vtk_widget = QVTKRenderWindowInteractor(self.frame)
        self.layout.addWidget(self.vtk_widget)
        
        # Controls
        self.controls_layout = QHBoxLayout()
        # Slider's label
        self.slider_label = QLabel("Area Percent Change:")
        self.controls_layout.addWidget(self.slider_label)
        # Slider for aneurysm area increase
        self.area_slider = QSlider(Qt.Orientation.Horizontal)
        self.area_slider.setRange(100, 1000)
        self.area_slider.setValue(500)
        self.controls_layout.addWidget(self.area_slider)
        # Button for running the deformation
        self.run_button = QPushButton("Run Deformation")
        self.controls_layout.addWidget(self.run_button)
        # Add controls layout to main layout
        self.layout.addLayout(self.controls_layout)
        self.frame.setLayout(self.layout)
        self.setCentralWidget(self.frame)
        # Connect the button to the run_deformation method
        self.run_button.clicked.connect(self.run_deformation)
        
        # VTK Setup
        self.vtk_interactor = self.vtk_widget.GetRenderWindow().GetInteractor()
        self.ren = vtk.vtkRenderer()
        self.vtk_widget.GetRenderWindow().AddRenderer(self.ren)
        
        self.initialize_vtk()

    def initialize_vtk(self):
        # Load initial VTK files and set up the scene here
        # Example:
        # self.mesh = load_vtp_file("mesh-complete-exterior.vtp")
        # self.centerline = load_vtp_file("centerline.vtp")
        # self.ren.AddActor(mesh_actor)
        # self.ren.AddActor(centerline_actor)
        # self.ren.ResetCamera()
        pass

    def run_deformation(self):
        area_percent_change = self.area_slider.value()
        print(f"Running deformation with area percent change: {area_percent_change}")
        # Trigger the deformation logic here
        # Example:
        # create_aneurysm(self.mesh, self.centerline, self.selected_points, area_percent_change)
        # self.update_vtk_view()
    
    def update_vtk_view(self):
        # Update VTK view after deformation
        self.vtk_widget.GetRenderWindow().Render()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())


: 

In [1]:
###### Full VTK Integration ######
import sys
from PyQt6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget, QPushButton, QSlider, QLabel, QHBoxLayout, QFileDialog, QLineEdit
from PyQt6.QtCore import Qt
import vtkmodules.all as vtk
from vtk.qt.QVTKRenderWindowInteractor import QVTKRenderWindowInteractor
from vtk_module import VTKHandler

class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super(MainWindow, self).__init__(parent)
        
        self.setWindowTitle("VTK with PyQt6")
        
        self.frame = QWidget()
        self.layout = QVBoxLayout()
        
        # Set the window size to be 800x600 pixels and position it at 100, 100
        self.setGeometry(100, 100, 1200, 900)
        
        # VTK Render Widget
        self.vtk_widget = QVTKRenderWindowInteractor(self.frame)
        self.layout.addWidget(self.vtk_widget)
        
        # Controls
        self.controls_layout = QHBoxLayout()

        # Import Buttons
        self.import_mesh_button = QPushButton("Import Mesh")
        self.import_mesh_button.clicked.connect(self.import_mesh)
        self.controls_layout.addWidget(self.import_mesh_button)
        self.import_centerline_button = QPushButton("Import Centerline")
        self.import_centerline_button.clicked.connect(self.import_centerline)
        self.controls_layout.addWidget(self.import_centerline_button)

        # Slider's label
        self.slider_label = QLabel("Area Percent Change:")
        self.controls_layout.addWidget(self.slider_label)
        # Slider for aneurysm area increase
        self.area_slider = QSlider(Qt.Orientation.Horizontal)
        self.area_slider.setRange(100, 1000)
        self.area_slider.setValue(500.0)
        self.controls_layout.addWidget(self.area_slider)
        # Display the slider value
        self.slider_value = QLineEdit()
        self.slider_value.setText(f"{self.area_slider.value()}%")
        self.slider_value.setFixedWidth(50)
        self.controls_layout.addWidget(self.slider_value)
        self.area_slider.valueChanged.connect(lambda value: self.slider_value.setText(f"{value}%"))
        self.slider_value.textChanged.connect(lambda text: self.area_slider.setValue(float(text.replace("%", ""))))
        # Button for showing selectable nodes on the centerline
        self.show_nodes_button = QPushButton("Select Nodes")
        self.controls_layout.addWidget(self.show_nodes_button)
        # Button for running the deformation
        self.run_button = QPushButton("Create Aneurysm")
        self.controls_layout.addWidget(self.run_button)
        # Add controls layout to main layout
        self.layout.addLayout(self.controls_layout)
        self.frame.setLayout(self.layout)
        self.setCentralWidget(self.frame)
        # Connect the button to the run_deformation method
        self.run_button.clicked.connect(self.run_deformation)
        # Connect the button to the show_nodes method
        self.show_nodes_button.clicked.connect(self.display_centerline_nodes)

        # To add a second row of buttons, add another QHBoxLayout and add it to the main layout
        self.controls_layout2 = QHBoxLayout()
        # Add stenosis controls
        self.stenosis_slider_label = QLabel("Stenosis Area % Change:")
        self.controls_layout2.addWidget(self.stenosis_slider_label)
        
        self.stenosis_area_slider = QSlider(Qt.Orientation.Horizontal)
        self.stenosis_area_slider.setRange(1, 100)
        self.stenosis_area_slider.setValue(5)
        self.controls_layout2.addWidget(self.stenosis_area_slider)
        
        self.stenosis_slider_value = QLineEdit()
        self.stenosis_slider_value.setText(f"{self.stenosis_area_slider.value()}")
        self.stenosis_slider_value.setFixedWidth(50)
        self.controls_layout2.addWidget(self.stenosis_slider_value)
        
        self.num_ring_points_label = QLabel("Num Ring Points:")
        self.controls_layout2.addWidget(self.num_ring_points_label)
        
        self.num_ring_points_slider = QSlider(Qt.Orientation.Horizontal)
        self.num_ring_points_slider.setRange(1, 100)
        self.num_ring_points_slider.setValue(35)
        self.controls_layout2.addWidget(self.num_ring_points_slider)
        
        self.num_ring_points_value = QLineEdit()
        self.num_ring_points_value.setText(f"{self.num_ring_points_slider.value()}")
        self.num_ring_points_value.setFixedWidth(50)
        self.controls_layout2.addWidget(self.num_ring_points_value)
        
        self.run_stenosis_button = QPushButton("Create Stenosis")
        self.controls_layout2.addWidget(self.run_stenosis_button)
        self.layout.addLayout(self.controls_layout2)
        # Connect the slider and QLineEdit for stenosis area
        self.stenosis_area_slider.valueChanged.connect(lambda value: self.stenosis_slider_value.setText(f"{value}"))
        self.stenosis_slider_value.textChanged.connect(lambda text: self.stenosis_area_slider.setValue(int(text)))
        # Connect the slider and QLineEdit for num ring points
        self.num_ring_points_slider.valueChanged.connect(lambda value: self.num_ring_points_value.setText(f"{value}"))
        self.num_ring_points_value.textChanged.connect(lambda text: self.num_ring_points_slider.setValue(int(text)))
        # Connect the button to the run_stenosis method
        self.run_stenosis_button.clicked.connect(self.run_stenosis)
        
        # VTK Setup
        self.vtk_interactor = self.vtk_widget.GetRenderWindow().GetInteractor()
        self.vtk_handler = None
        # VTKHandler("input/mesh-complete-exterior.vtp", "input/centerline.vtp")

    def import_mesh(self):
        file_name, _ = QFileDialog.getOpenFileName(self, "Import Mesh", "", "VTK Files (*.vtp)")
        if file_name:
            self.mesh_file = file_name
            if hasattr(self, 'centerline_file'):
                self.initialize_vtk_handler()

    def import_centerline(self):
        file_name, _ = QFileDialog.getOpenFileName(self, "Import Centerline", "", "VTK Files (*.vtp)")
        if file_name:
            self.centerline_file = file_name
            if hasattr(self, 'mesh_file'):
                self.initialize_vtk_handler()

    def initialize_vtk_handler(self):
        self.vtk_handler = VTKHandler(self.mesh_file, self.centerline_file)
        self.ren = self.vtk_handler.get_renderer()
        self.vtk_widget.GetRenderWindow().AddRenderer(self.ren)

        self.style = self.vtk_handler.get_interactor_style(self.vtk_interactor)
        self.vtk_interactor.SetInteractorStyle(self.style)

        # self.vtk_widget.GetRenderWindow().Render()

        self.vtk_interactor.Initialize()
        self.vtk_interactor.Start()

    def run_deformation(self):
        area_percent_change = self.area_slider.value()
        print(f"Running deformation with area percent change: {area_percent_change}")
        self.style.deform_mesh(area_percent_change)
        # self.vtk_handler.get_interactor_style(self.vtk_interactor).deform_mesh()
        # self.vtk_widget.GetRenderWindow().Render()
    
    def run_stenosis(self):
        if self.vtk_handler is None:
            print("Please import both mesh and centerline files before running the stenosis.")
            return

        area_percent_change = self.stenosis_area_slider.value()
        num_ring_points = self.num_ring_points_slider.value()
        falloff_type = "regular"
        weight_regularized_laplacian = 1        
        print(f"Running stenosis with area percent change: {area_percent_change}, num ring points: {num_ring_points}")
        self.style.deform_mesh_stenosis(area_percent_change, num_ring_points, falloff_type, weight_regularized_laplacian)

    def display_centerline_nodes(self):
        print("Please select three centerline nodes to generate aneurysm.")
        self.style.display_vertices()
    
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())


/tmp/ipykernel_2063660/3082153310.py:42: DeprecationWarning: an integer is required (got type float).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  self.area_slider.setValue(500.0)


Please select three centerline nodes to generate aneurysm.
Picked actor centerpointID: 382
Picked actor centerpointID: 402
Picked actor centerpointID: 434
Running stenosis with area percent change: 5, num ring points: 35
---------------------------------------------------------------------- it =  0
ring point indices =  [0, 9, 18, 27, 36, 45, 54, 63, 72, 81, 90, 99, 108, 117, 126, 135, 144, 153, 162, 171, 180, 189, 198, 207, 216, 225, 234, 243, 252, 261, 270, 279, 288, 297, 306]
message =  Optimization terminated successfully.
true scale =  0.9807890588296149
percent difference =  -0.005490486511554755


2024-11-04 14:34:36.580 (  17.315s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 14:34:36.609 (  17.344s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 14:34:36.637 (  17.372s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 14:34:36.837 (  17.572s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter


ring point indices =  [0, 9, 18, 27, 36, 45, 54, 63, 72, 81, 90, 99, 108, 117, 126, 135, 144, 153, 162, 171, 180, 189, 198, 207, 216, 225, 234, 243, 252, 261, 270, 279, 288, 297, 306]
message =  Optimization terminated successfully.
true scale =  0.9807890588296149
percent difference =  -0.005490486511554755


2024-11-04 14:34:37.092 (  17.827s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 14:34:37.122 (  17.858s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 14:34:37.150 (  17.885s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter


...

ring point indices =  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
message =  Optimization terminated successfully.
true scale =  1.1432570598819538
percent difference =  0.0014988804946613474
---------------------------------------------------------------------- it =  5


2024-11-04 14:34:39.049 (  19.784s) [    7FF40CAB9280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter


Exception: Empty slice

SystemExit: 0

/home/bohanjeffli/miniconda3/envs/kelvinlet/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
# Profiling to check the bottleneck
import cProfile
import pstats
from functools import wraps

def profile_func(func):
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        profiler = cProfile.Profile()
        profiler.enable()
        result = func(self)
        profiler.disable()
        
        # Print profiling results
        ps = pstats.Stats(profiler)
        ps.strip_dirs().sort_stats("cumulative").print_stats(10)
        
        return result
    return wrapper

###### Full VTK Integration ######
import sys
from PyQt6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget, QPushButton, QSlider, QLabel, QHBoxLayout, QFileDialog, QLineEdit
from PyQt6.QtCore import Qt
import vtkmodules.all as vtk
from vtk.qt.QVTKRenderWindowInteractor import QVTKRenderWindowInteractor
from vtk_module import VTKHandler

class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super(MainWindow, self).__init__(parent)
        
        self.setWindowTitle("VTK with PyQt6")
        
        self.frame = QWidget()
        self.layout = QVBoxLayout()
        
        # Set the window size to be 800x600 pixels and position it at 100, 100
        self.setGeometry(100, 100, 1200, 900)
        
        # VTK Render Widget
        self.vtk_widget = QVTKRenderWindowInteractor(self.frame)
        self.layout.addWidget(self.vtk_widget)
        
        # Controls
        self.controls_layout = QHBoxLayout()

        # Import Buttons
        self.import_mesh_button = QPushButton("Import Mesh")
        self.import_mesh_button.clicked.connect(self.import_mesh)
        self.controls_layout.addWidget(self.import_mesh_button)
        self.import_centerline_button = QPushButton("Import Centerline")
        self.import_centerline_button.clicked.connect(self.import_centerline)
        self.controls_layout.addWidget(self.import_centerline_button)

        # Slider's label
        self.slider_label = QLabel("Area Percent Change:")
        self.controls_layout.addWidget(self.slider_label)
        # Slider for aneurysm area increase
        self.area_slider = QSlider(Qt.Orientation.Horizontal)
        self.area_slider.setRange(100, 1000)
        self.area_slider.setValue(500.0)
        self.controls_layout.addWidget(self.area_slider)
        # Display the slider value
        self.slider_value = QLineEdit()
        self.slider_value.setText(f"{self.area_slider.value()}%")
        self.slider_value.setFixedWidth(50)
        self.controls_layout.addWidget(self.slider_value)
        self.area_slider.valueChanged.connect(lambda value: self.slider_value.setText(f"{value}%"))
        self.slider_value.textChanged.connect(lambda text: self.area_slider.setValue(float(text.replace("%", ""))))
        # Button for showing selectable nodes on the centerline
        self.show_nodes_button = QPushButton("Select Nodes")
        self.controls_layout.addWidget(self.show_nodes_button)
        # Button for running the deformation
        self.run_button = QPushButton("Create Aneurysm")
        self.controls_layout.addWidget(self.run_button)
        # Add controls layout to main layout
        self.layout.addLayout(self.controls_layout)
        self.frame.setLayout(self.layout)
        self.setCentralWidget(self.frame)
        # Connect the button to the run_deformation method
        self.run_button.clicked.connect(self.run_deformation)
        # Connect the button to the show_nodes method
        self.show_nodes_button.clicked.connect(self.display_centerline_nodes)

        # To add a second row of buttons, add another QHBoxLayout and add it to the main layout
        self.controls_layout2 = QHBoxLayout()
        # Add stenosis controls
        self.stenosis_slider_label = QLabel("Stenosis Area % Change:")
        self.controls_layout2.addWidget(self.stenosis_slider_label)
        
        self.stenosis_area_slider = QSlider(Qt.Orientation.Horizontal)
        self.stenosis_area_slider.setRange(1, 100)
        self.stenosis_area_slider.setValue(5)
        self.controls_layout2.addWidget(self.stenosis_area_slider)
        
        self.stenosis_slider_value = QLineEdit()
        self.stenosis_slider_value.setText(f"{self.stenosis_area_slider.value()}")
        self.stenosis_slider_value.setFixedWidth(50)
        self.controls_layout2.addWidget(self.stenosis_slider_value)
        
        self.num_ring_points_label = QLabel("Num Ring Points:")
        self.controls_layout2.addWidget(self.num_ring_points_label)
        
        self.num_ring_points_slider = QSlider(Qt.Orientation.Horizontal)
        self.num_ring_points_slider.setRange(1, 100)
        self.num_ring_points_slider.setValue(35)
        self.controls_layout2.addWidget(self.num_ring_points_slider)
        
        self.num_ring_points_value = QLineEdit()
        self.num_ring_points_value.setText(f"{self.num_ring_points_slider.value()}")
        self.num_ring_points_value.setFixedWidth(50)
        self.controls_layout2.addWidget(self.num_ring_points_value)
        
        self.run_stenosis_button = QPushButton("Create Stenosis")
        self.controls_layout2.addWidget(self.run_stenosis_button)
        self.layout.addLayout(self.controls_layout2)
        # Connect the slider and QLineEdit for stenosis area
        self.stenosis_area_slider.valueChanged.connect(lambda value: self.stenosis_slider_value.setText(f"{value}"))
        self.stenosis_slider_value.textChanged.connect(lambda text: self.stenosis_area_slider.setValue(int(text)))
        # Connect the slider and QLineEdit for num ring points
        self.num_ring_points_slider.valueChanged.connect(lambda value: self.num_ring_points_value.setText(f"{value}"))
        self.num_ring_points_value.textChanged.connect(lambda text: self.num_ring_points_slider.setValue(int(text)))
        # Connect the button to the run_stenosis method
        self.run_stenosis_button.clicked.connect(self.run_stenosis)
        
        # VTK Setup
        self.vtk_interactor = self.vtk_widget.GetRenderWindow().GetInteractor()
        self.vtk_handler = None
        # VTKHandler("input/mesh-complete-exterior.vtp", "input/centerline.vtp")

    def import_mesh(self):
        file_name, _ = QFileDialog.getOpenFileName(self, "Import Mesh", "", "VTK Files (*.vtp)")
        if file_name:
            self.mesh_file = file_name
            if hasattr(self, 'centerline_file'):
                self.initialize_vtk_handler()

    def import_centerline(self):
        file_name, _ = QFileDialog.getOpenFileName(self, "Import Centerline", "", "VTK Files (*.vtp)")
        if file_name:
            self.centerline_file = file_name
            if hasattr(self, 'mesh_file'):
                self.initialize_vtk_handler()

    def initialize_vtk_handler(self):
        self.vtk_handler = VTKHandler(self.mesh_file, self.centerline_file)
        self.ren = self.vtk_handler.get_renderer()
        self.vtk_widget.GetRenderWindow().AddRenderer(self.ren)

        self.style = self.vtk_handler.get_interactor_style(self.vtk_interactor)
        self.vtk_interactor.SetInteractorStyle(self.style)

        # self.vtk_widget.GetRenderWindow().Render()

        self.vtk_interactor.Initialize()
        self.vtk_interactor.Start()

    def run_deformation(self):
        area_percent_change = self.area_slider.value()
        print(f"Running deformation with area percent change: {area_percent_change}")
        self.style.deform_mesh(area_percent_change)
        # self.vtk_handler.get_interactor_style(self.vtk_interactor).deform_mesh()
        # self.vtk_widget.GetRenderWindow().Render()
    
    @profile_func
    def run_stenosis(self):
        if self.vtk_handler is None:
            print("Please import both mesh and centerline files before running the stenosis.")
            return

        area_percent_change = self.stenosis_area_slider.value()
        num_ring_points = self.num_ring_points_slider.value()
        falloff_type = "regular"
        weight_regularized_laplacian = 1        
        print(f"Running stenosis with area percent change: {area_percent_change}, num ring points: {num_ring_points}")
        self.style.deform_mesh_stenosis(area_percent_change, num_ring_points, falloff_type, weight_regularized_laplacian)

    def display_centerline_nodes(self):
        print("Please select three centerline nodes to generate aneurysm.")
        self.style.display_vertices()
    
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())


/tmp/ipykernel_2181263/1371474759.py:62: DeprecationWarning: an integer is required (got type float).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  self.area_slider.setValue(500.0)


Please select three centerline nodes to generate aneurysm.
Picked actor centerpointID: 371
Picked actor centerpointID: 401
Picked actor centerpointID: 432
Running stenosis with area percent change: 5, num ring points: 35
---------------------------------------------------------------------- it =  0
ring point indices =  [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300, 310, 320, 330, 340]
message =  Optimization terminated successfully.
true scale =  0.9807890605194527
percent difference =  -0.00549031422762953


2024-11-04 15:55:54.755 (  16.074s) [    7FB6C0101280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 15:55:54.788 (  16.107s) [    7FB6C0101280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-11-04 15:55:54.818 (  16.137s) [    7FB6C0101280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter


kk.shape =  (21662, 35, 3, 3)
kk.shape =  (618, 35, 3, 3)
ring point indices =  [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300, 310, 320, 330, 340]
message =  Optimization terminated successfully.
true scale =  0.9807890605194527
percent difference =  -0.00549031422762953


2024-11-04 15:55:55.050 (  16.369s) [    7FB6C0101280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter


...

kk.shape =  (21662, 35, 3, 3)
kk.shape =  (618, 35, 3, 3)


2024-11-04 15:56:07.253 (  28.573s) [    7FB6C0101280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter


         671982 function calls (662529 primitive calls) in 13.140 seconds

   Ordered by: cumulative time
   List reduced from 289 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   13.140   13.140 1371474759.py:166(run_stenosis)
        1    0.000    0.000   13.140   13.140 vtk_module.py:164(deform_mesh_stenosis)
        1    0.000    0.000   13.077   13.077 vtk_module.py:239(create_stenosis)
        1    0.032    0.032   13.077   13.077 scaling.py:571(run_stenosis_v4)
      100    0.076    0.001    9.817    0.098 scaling.py:315(get_ring_displacements_v2)
       75    3.357    0.045    4.289    0.057 common.py:82(kelvinlets_translation_v2)
       75    3.067    0.041    4.152    0.055 common.py:212(laplacian_kelvinlets_translation_v2)
85758/80882    0.076    0.000    2.647    0.000 {built-in method numpy.core._multiarray_umath.implement_array_function}
      312    2.105    0.007    2.105    0.007 {metho

SystemExit: 0

/home/bohanjeffli/miniconda3/envs/kelvinlet/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [21]:
import numpy as np

# Assume kk is of shape (num_mesh_points, num_ring_points, 3, 3)
# Reshape it to (num_mesh_points, num_ring_points, 3 * num_ring_points, 3)
kk_example = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9], [1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8, 9.9]] * 1).reshape(1, 2, 3, 3)
print(kk_example)
# to construct a block diagonal matrix of shape (num_mesh_points, 3 * num_ring_points, num_mesh_points, 3 * num_ring_points), we can do
kelvinlet_matrix_example = kk_example.transpose(0, 2, 1, 3).reshape(1, 3, 3 * 2)
# to permute a numpy array, we can use np.transpose

print(kelvinlet_matrix_example)

[[[[1.  2.  3. ]
   [4.  5.  6. ]
   [7.  8.  9. ]]

  [[1.1 2.2 3.3]
   [4.4 5.5 6.6]
   [7.7 8.8 9.9]]]]
[[[1.  2.  3.  1.1 2.2 3.3]
  [4.  5.  6.  4.4 5.5 6.6]
  [7.  8.  9.  7.7 8.8 9.9]]]
